# 🔬 Asset Analytics — Correlations, Cointegration, Comparison
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/05_asset_analytics.ipynb)

Correlation matrices (static and rolling), return autocorrelation, Engle-Granger cointegration, a return-based screener, PCA, and side-by-side fund comparison on daily data.

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Settings {display-mode: "form"}
tickers = "SPY, QQQ, TLT, GLD, VNQ, EFA, EEM, BTC-USD"  #@param {type:"string"}
start_date = "2015-01-01"  #@param {type:"date"}
rolling_corr_pair = "SPY, TLT"  #@param {type:"string"}
rolling_window_days = 63   #@param {type:"number"}
cointegration_pair = "SPY, IVV"  #@param {type:"string"}
SMOKE = False

In [ ]:
from portlab import analytics, plots
from portlab.data import get_prices, get_returns

tick_list = [t.strip().upper() for t in tickers.split(",") if t.strip()]
rets = get_returns(tick_list, start_date)
plots.corr_heatmap(analytics.correlation_matrix(rets)).show()
analytics.performance_table(rets).style.format("{:.3f}")

In [ ]:
a, b = [t.strip().upper() for t in rolling_corr_pair.split(",")]
rc = analytics.rolling_correlation(rets[a], rets[b], window=rolling_window_days)
plots.rolling_chart(rc.dropna().rename(f"{a} vs {b}"),
                    f"Rolling {rolling_window_days}-Day Correlation", yformat=".2f").show()

In [ ]:
ac = analytics.autocorrelation(rets[tick_list[0]], lags=20)
print(f"95% CI: ±{ac.attrs['ci95']:.3f}")
ac.to_frame("autocorrelation").style.format("{:.4f}")

In [ ]:
ca, cb = [t.strip().upper() for t in cointegration_pair.split(",")]
cp = get_prices([ca, cb], start_date)
analytics.cointegration_test(cp[ca], cp[cb])

In [ ]:
#@title Fund screener — return-based metrics over any universe {display-mode: "form"}
screen_universe = "SPY, QQQ, VTV, VUG, SCHD, USMV, MTUM, QUAL, IWM, EFA, EEM, VNQ, TLT, HYG"  #@param {type:"string"}
min_sharpe = 0.3   #@param {type:"number"}
max_drawdown_worse_than = -0.60  #@param {type:"number"}
sort_metric = "Sharpe Ratio"  #@param ["Sharpe Ratio", "CAGR", "Sortino Ratio", "Max Drawdown", "Annualized Volatility"]
u = [t.strip().upper() for t in screen_universe.split(",") if t.strip()]
urets = get_returns(u, start_date)
analytics.screener(urets, filters={"Sharpe Ratio": (min_sharpe, None),
                                   "Max Drawdown": (max_drawdown_worse_than, None)},
                   sort_by=sort_metric).style.format("{:.3f}")

In [ ]:
# Principal component analysis — the statistical factors driving this universe
load = analytics.pca(rets, n_components=4)
print("Explained variance:", ", ".join(f"{k}: {v:.0%}" for k, v in load.attrs["explained_variance"].items()))
plots.corr_heatmap(load.T, "PCA Loadings by Asset").show()